In [ ]:
!pip install -U scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import data_utils_clean
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer,KNNImputer,MissingIndicator
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder,MinMaxScaler,PowerTransformer,OrdinalEncoder
from sklearn.model_selection import train_test_split

In [ ]:
!pip install dagshub

In [ ]:
import dagshub
dagshub.init(repo_owner='sneha12603', repo_name='Delivery-time-prediction', mlflow=True)

In [ ]:
!pip install mlflow

In [ ]:
import mlflow

In [ ]:
#set the tracking server
mlflow.set_tracking_uri("https://dagshub.com/sneha12603/Delivery-time-prediction.mlflow")

In [ ]:
mlflow.set_experiment("Exp 3 - Model Selection")

In [ ]:
#load the data
from sklearn import set_config

set_config(transform_output="pandas")

In [ ]:
df = pd.read_csv("swiggy.csv")
df

In [ ]:
#clean the data
data_utils_clean.perform_data_cleaning(df)

In [ ]:
#load the cleaned data
df = pd.read_csv('swiggy_cleaned.csv')

In [ ]:
df.columns

In [ ]:
#drop columns not required for model output
columns_to_drop = ['restaurant_longitude',
                   'order_time_hour', # This column was dropped during cleaning
                   'city_name', # This column was dropped during cleaning
                   'order_day_of_week', # This column was dropped during cleaning
                   'order_month' # This column was dropped during cleaning
                   ]

df.drop(columns=columns_to_drop,inplace=True)
df

In [ ]:
#check for missing values
df.isna().sum()

In [ ]:
#check for duplicates
df.duplicated().sum()

In [ ]:
import missingno as msno
msno.matrix(df)

In [ ]:
#column that have missing values
missing_cols = (
    df.isna().any(axis=0).loc[lambda x: x].index
)

missing_cols

In [ ]:
#drop missing values
temp_df = df.copy().dropna()

In [ ]:
#split into X and y
X = temp_df.drop(columns='time_taken')
y = temp_df['time_taken']

In [ ]:
X

In [ ]:
#train test split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
print("The size of train data is",X_train.shape)
print("The size of test data is",X_test.shape)

In [ ]:
X_train.isna().sum()

In [ ]:
#transorm target column
pt = PowerTransformer()

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

In [ ]:
missing_cols

In [ ]:
#percentage of rows in dat having missing values
(
    X_train.isna().any(axis=1).mean().round(2)*100
)

***Pre Processing Pipeline***

In [ ]:
num_cols = ["age","ratings","pickup_times","distance"]

nominal_cat_cols = ['weather',
                   'type_of_order',
                   'type_of_vehicle',
                   'festival',
                   'city_type',
                   'is_weekend',
                   'order_time_of_day']

ordinal_cat_cols = ['traffic','distance_type']

In [ ]:
nominal_cat_cols

In [ ]:
X_train.isna().sum()

In [ ]:
#do basic preprocessing
num_cols = ['age','ratings','pickup_time_minutes','distance']

nominal_cat_cols = ['weather','type_of_order',
                    'type_of_vehicle','festival',
                    'city_type','is_weekend',
                    'order_time_of_day']

ordinal_cat_cols = ['traffic','distance_type']

In [ ]:
#genertate order for ordinal encoding
traffic_order = ['low','medium','high','jam']

distance_type_order = ['short','medium','long','very_long']

In [ ]:
#unique categories in ordinal column
for col in ordinal_cat_cols:
  print(col,X_train[col].unique())

In [ ]:
#build a preprocessor
preprocessor = ColumnTransformer(transformers=[
    ("scale",MinMaxScaler(),num_cols),
    ("nominal_encode",OneHotEncoder(drop='first',handle_unknown='ignore',
                                    sparse_output=False),nominal_cat_cols),
    ("ordinal_encode",OrdinalEncoder(categories=[traffic_order,distance_type_order],
                                     encoded_missing_value = -999,
                                     handle_unknown='use_encoded_value',
                                     unknown_value=-1),ordinal_cat_cols)

],remainder='drop',n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)

preprocessor

In [ ]:

#build the pipeline
processing_pipeline = Pipeline(steps=[
    ("preprocess",preprocessor)

])

In [ ]:
#data preprocessing
X_train_trans = processing_pipeline.fit_transform(X_train)
X_test_trans = processing_pipeline.transform(X_test)

In [ ]:
X_train_trans

In [ ]:
!pip install optuna

In [ ]:
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna

In [ ]:
from sklearn.metrics import r2_score,mean_absolute_error

In [ ]:
def objective(trial):
  with mlflow.start_run(nested=True):
    model_name = trial.suggest_categorical("model",["SVM","RF","KNN","GB","XGB","LGBM"])

    if model_name == "SVM":
      kernel_svm = trial.suggest_categorical("kernal_svm",["linear","ploy","rbf"])
      if kernel_svm == "linear":
        c_linear = trial.suggest_float("c_linear",0,10)
        model = SVR(C=c_linear,kernel="linear")

      elif kernel_svm == "poly":
        c_poly = trial.suggest_float("c_poly",0,10)
        degree_poly = trial.suggest_int("degree_poly",1,5)
        model = SVR(C=c_poly,degree = degree_poly,
                    kernel="rbf")

      else:
        c_rbf = trial.suggest_float("c_rbf",0,100)
        gamma_rbf = trial.suggest_float("gamma_rbf",0,10)
        model = SVR(C=c_rbf,gamma = gamma_rbf,
                    kernel="rbf")

    elif model_name == "RF":
      n_estimators_rf = trial.suggest_int("n_estimators_rf",10,200)
      max_depth_rf = trial.suggest_int("max_depth_rf",2,20)
      model = RandomForestRegressor(n_estimators=n_estimators_rf,
                                    max_depth=max_depth_rf,
                                    n_jobs=-1)
    elif model_name == "GB":
      n_estimators_gb = trial.suggest_int("n_estimators_gb",10,200)
      learning_rate_gb = trial.suggest_float("learning_rate_gb",0,1)
      max_depth_gb = trial.suggest_int("max_depth_gb",2,20)
      model = GradientBoostingRegressor(n_estimators=n_estimators_gb,
                                        learning_rate = learning_rate_gb,
                                        max_depth = max_depth_gb,
                                        random_state=42)


    elif model_name == "KNN":
      n_neighbors_knn = trial.suggest_int("n_neighbors_knn",1,25)
      weights_knn = trial.suggest_categorical("weights_knn",["uniform","distance"])
      model = KNeighborsRegressor(n_neighbors=n_neighbors_knn,
                                  weights=weights_knn,n_jobs=-1)


    elif model_name == "XGB":
      n_estimators_xgb = trial.suggest_int("n_estimators_xgb",10,200)
      learning_rate_xgb = trial.suggest_float("learning_rate_xgb",0.1,0.5)
      max_depth_xgb = trial.suggest_int("max_depth_xgb",2,20)
      model = XGBRegressor(n_estimators=n_estimators_xgb,
                         learning_rate = learning_rate_xgb,
                         max_depth = max_depth_xgb,
                         random_state=42,
                         n_jobs=-1)

    elif model_name == "LGBM":
     n_estimators_lgbm = trial.suggest_int("n_estmators_lgbm",10,200)
     learning_rate_lgbm = trial.suggest_float("learning_rate_lgbm",0.1,0.5)
     max_depth_lgbm = trial.suggest_int("max_depth_lgbm",2,20)
     model = LGBMRegressor(n_estimators=n_estimators_lgbm,
                           learning_rate=learning_rate_lgbm,
                           max_depth=max_depth_lgbm,
                           random_state=42)


    #train the model
    model.fit(X_train_trans,y_train_pt.values.ravel())

    #log model params
    mlflow.log_params(model.get_params())

    #get the predictions
    y_pred_train = model.predict(X_train_trans)
    y_pred_test = model.predict(X_test_trans)

    #get the actual prediction values
    y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
    y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))

    #calculate the error
    error = mean_absolute_error(y_test,y_pred_test_org)

    #log model name
    mlflow.log_param("model",model_name)

    #log error
    mlflow.log_metric("MAE",error)

    return error

In [ ]:
#create optuna study
study = optuna.create_study(direction="minimize",study_name="model_selection")

with mlflow.start_run(run_name="Best Model") as parent:
  study.optimize(objective,n_trials=30,n_jobs=-1)

  #log the best parameters
  mlflow.log_params(study.best_params)

  #log the best score
  mlflow.log_metric("best_score",study.best_value)

In [ ]:
study.best_value

In [ ]:
lgbm_params = {
    "n_estimators":151,
    "learning_rate":0.14553773585462942,
    "max_depth":20
}

In [ ]:
#train the model on best parameters
lgbm = LGBMRegressor(**lgbm_params)
lgbm.fit(X_train_trans,y_train_pt.values.ravel())


In [ ]:
#get the predictions
y_pred_train = lgbm.predict(X_train_trans)
y_pred_test = lgbm.predict(X_test_trans)



In [ ]:
#get the actual predictions values
y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))


In [ ]:
from sklearn.metrics import mean_absolute_error,r2_score

print(f"train r2 score is {r2_score(y_train,y_pred_train_org):.2f}")
print(f"test score is {r2_score(y_test,y_pred_test_org):.2f}")

In [ ]:
print(f"train error is {mean_absolute_error(y_train,y_pred_train_org):.2f}minutes")
print(f"test error is {mean_absolute_error(y_test,y_pred_test_org):.2f}minutes")


In [ ]:
#dataframe of results
study.trials_dataframe()

In [ ]:
#model frequency
study.trials_dataframe()['params_model'].value_counts()

In [ ]:
#avg scores for all tested models
study.trials_dataframe().groupby("params_model")['value'].mean().sort_values()

In [ ]:
from sklearn.compose import TransformedTargetRegressor
model = TransformedTargetRegressor(regressor=lgbm,transformer=pt)

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X_train_trans,y_train,scoring="neg_mean_absolute_error",cv=5,n_jobs=-1)
scores

In [ ]:
#means score
-scores.mean()

In [ ]:
#optimization history plot
optuna.visualization.plot_optimization_history(study)

In [ ]:
#partial co-ordinate plot
optuna.visualization.plot_parallel_coordinate(study,params=["model"])